# 05 · Additive XGBoost

**Competition:** Kaggle Playground Series S6E9 — Predicting Electric Vehicle Purchases  
**Metric:** ROC-AUC

This notebook develops a structurally constrained XGBoost model designed to test whether restricting arbitrary cross-field interactions can improve generalization.

The model uses a high-resolution feature matrix while grouping engineered features by their original source field.

The pipeline combines:

1. all raw competition features;
2. six annual-income digit features;
3. income quantization at `//10`, `//100`, and `//1000`;
4. integer commute and income-boundary indicators;
5. fold-local frequency encoding;
6. three cross-fitted Target Encodings (`auto`, `10`, `100`);
7. outer 10-fold / inner 10-fold validation;
8. histogram-based XGBoost with `max_bin=16384`;
9. shallow trees with explicit regularization;
10. interaction constraints grouped by original feature ownership.

The interaction constraints allow different representations of the same original field to interact within a tree while preventing arbitrary cross-field tree interactions.

This model tests a structural regularization hypothesis: much of the remaining predictive signal may be captured by strong nonlinear effects within individual fields without requiring unrestricted cross-field tree interactions.

## 1. Imports and configuration

The notebook expects:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

The original successful run used Kaggle GPU training.

For compatibility with current scikit-learn, `TargetEncoder` is configured with integer `cv=10`, `shuffle=True`, and `random_state=42`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import xgboost as xgb

from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = Path(".")
OUTPUT_DIR = Path("outputs/05_additive_xgb")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Experiment setup
# -----------------------------
TARGET = "Will_Buy_EV"
ID_COL = "id"
INCOME = "Annual_Income_USD"
COMMUTE = "Daily_Commute_km"

N_FOLDS = 10
INNER_FOLDS = 10
SEED = 42
THREADS = 8

DEVICE = "cuda"
MAX_BIN = 16384
ROUNDS = 20000
PATIENCE = 500

CATS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

NUMS = [
    "Age",
    INCOME,
    COMMUTE,
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

RAW = NUMS + CATS

## 2. Load data

Categorical variables are factorized into integer codes because XGBoost receives the engineered matrix as numeric input.

The target is mapped from `No/Yes` to `0/1`.

In [ ]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    usecols=[ID_COL, TARGET] + RAW,
)

test = pd.read_csv(
    DATA_DIR / "test.csv",
    usecols=[ID_COL] + RAW,
)

sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv",
    usecols=[ID_COL, TARGET],
)

y = (
    train[TARGET]
    .map({"No": 0, "Yes": 1})
    .to_numpy(dtype=np.int8)
)

raw = pd.concat(
    [train[RAW], test[RAW]],
    ignore_index=True,
)

test_idx = np.arange(
    len(train),
    len(raw),
)

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Positive rate:", y.mean())

## 3. High-resolution feature matrix

Each engineered feature is assigned an **owner**: the original raw field from which it was derived.

That ownership map is later converted into XGBoost interaction constraints.

### 3.1 Raw features

All 13 raw model fields are retained. Categorical variables are factorized.

### 3.2 Income digits

Six integer digits are extracted:

$$
d_p(x)
=
\left\lfloor
\frac{x}{10^p}
\right\rfloor
\bmod 10,
\qquad
p=0,\ldots,5.
$$

### 3.3 Quantized income

Three additional income representations:

$$
\left\lfloor \frac{x}{10}\right\rfloor,\quad
\left\lfloor \frac{x}{100}\right\rfloor,\quad
\left\lfloor \frac{x}{1000}\right\rfloor.
$$

### 3.4 Additional structural variables

- `commute_integer`
- `income_30k`
- `income_cliff`
- `income_interval`

Every one of these engineered columns inherits ownership from its source raw field.

In [ ]:
features = {}
owner = {}


def add_feature(
    name,
    values,
    original_column,
):
    features[name] = np.asarray(
        values,
        dtype=np.float32,
    )
    owner[name] = original_column


# -----------------------------
# Raw features
# -----------------------------
for col in RAW:
    if col in CATS:
        values = pd.factorize(
            raw[col],
            sort=True,
        )[0]
    else:
        values = raw[col].to_numpy()

    add_feature(
        col,
        values,
        col,
    )


income = raw[INCOME].to_numpy(
    dtype=np.int64
)

# -----------------------------
# Six income digits
# -----------------------------
for power in range(6):
    add_feature(
        f"income_digit_{power}",
        (
            income
            // (10 ** power)
            % 10
        ),
        INCOME,
    )

# -----------------------------
# Income quantization
# -----------------------------
for divisor in [
    10,
    100,
    1000,
]:
    add_feature(
        f"income_floor_{divisor}",
        income // divisor,
        INCOME,
    )

# -----------------------------
# Commute / income structure
# -----------------------------
add_feature(
    "commute_integer",
    np.floor(
        raw[COMMUTE]
    ),
    COMMUTE,
)

add_feature(
    "income_30k",
    income == 30000,
    INCOME,
)

add_feature(
    "income_cliff",
    income >= 170537,
    INCOME,
)

add_feature(
    "income_interval",
    (
        (income >= 38000)
        & (income <= 42000)
    ),
    INCOME,
)

X = pd.DataFrame(features)

print("Base engineered features:", X.shape[1])

## 4. Encoding keys

Frequency encoding and Target Encoding use 17 keys:

- the 13 raw fields;
- `income_floor_10`;
- `income_floor_100`;
- `income_floor_1000`;
- `commute_integer`.

The six income digits are **model features**, but they are not separate Target-Encoding keys in this experiment.

In [ ]:
KEYS = RAW + [
    "income_floor_10",
    "income_floor_100",
    "income_floor_1000",
    "commute_integer",
]

assert len(KEYS) == 17

## 5. Fold-specific feature construction

For each outer fold:

### 5.1 Fold-local frequency encoding

For key $k$,

$$
\mathrm{freq}(v)
=
\frac{
\#\{i\in\mathrm{outer\ train}:k_i=v\}
}{
n_{\mathrm{outer\ train}}
}.
$$

Validation/test unseen values receive `0`.

### 5.2 Triple Target Encoding

For each of the 17 keys, three Target Encodings are generated:

- `smooth="auto"`
- `smooth=10`
- `smooth=100`

Training rows receive inner 10-fold cross-fitted encodings. Outer validation and test rows are transformed by encoders fitted only on outer-training labels.

### 5.3 Feature ownership

Every frequency and TE column inherits the owner of its source key. This produces the field-level groups used by `interaction_constraints`.

In [ ]:
def make_fold(
    train_idx,
    valid_idx,
):
    indices = [
        train_idx,
        valid_idx,
        test_idx,
    ]

    frames = [
        X.iloc[idx].copy()
        for idx in indices
    ]

    families = owner.copy()

    # -----------------------------
    # Fold-local frequencies
    # -----------------------------
    for col in KEYS:
        frequency = (
            X.iloc[train_idx][col]
            .value_counts(
                normalize=True
            )
        )

        for frame in frames:
            frame[
                f"freq_{col}"
            ] = (
                frame[col]
                .map(frequency)
                .fillna(0)
                .astype("float32")
            )

        families[
            f"freq_{col}"
        ] = owner[col]

    # -----------------------------
    # Triple cross-fitted TE
    # -----------------------------
    for smooth in [
        "auto",
        10.0,
        100.0,
    ]:
        encoder = TargetEncoder(
            target_type="binary",
            smooth=smooth,
            cv=INNER_FOLDS,
            shuffle=True,
            random_state=SEED,
        )

        encoded_train = (
            encoder.fit_transform(
                X.iloc[train_idx][KEYS],
                y[train_idx],
            )
        )

        encoded_valid = (
            encoder.transform(
                X.iloc[valid_idx][KEYS]
            )
        )

        encoded_test = (
            encoder.transform(
                X.iloc[test_idx][KEYS]
            )
        )

        columns = [
            f"TE_{col}_{smooth}"
            for col in KEYS
        ]

        for frame, values in zip(
            frames,
            [
                encoded_train,
                encoded_valid,
                encoded_test,
            ],
        ):
            frame[columns] = (
                values.astype(
                    "float32"
                )
            )

        families.update({
            feature_name: owner[key_name]
            for feature_name, key_name
            in zip(
                columns,
                KEYS,
            )
        })

    # Remove outer-train constants.
    keep = [
        col
        for col in frames[0].columns
        if frames[0][col].nunique() > 1
    ]

    frames = [
        frame[keep].reset_index(
            drop=True
        )
        for frame in frames
    ]

    # One group per original raw field.
    groups = [
        [
            i
            for i, col in enumerate(keep)
            if families[col] == source
        ]
        for source in RAW
    ]

    groups = [
        group
        for group in groups
        if group
    ]

    # Every model feature must belong to exactly one owner group.
    assert sorted(
        i
        for group in groups
        for i in group
    ) == list(
        range(len(keep))
    )

    return (
        frames,
        groups,
    )

## 6. Interaction constraints

The key structural choice is:

$$
\text{one interaction group per original raw field}.
$$

Suppose a tree first splits on `Annual_Income_USD`. Under these constraints, later splits in that tree can use other **income-owned** representations such as:

- income digits;
- income quantization;
- income frequency;
- income Target Encodings.

But that tree cannot then split on an unrelated original field such as `Gender` or `City_Type`.

The same rule applies independently to every original field.

This creates a strongly additive tree ensemble across source fields while still allowing nonlinear interaction among multiple representations of the **same** field.

The hypothesis was not that the true data-generating process was perfectly additive. It was that restricting arbitrary cross-field interaction could reduce noisy tree capacity while preserving the strongest one-field nonlinear structure.

## 7. XGBoost configuration

The successful model used:

- `tree_method="hist"`
- GPU device
- `max_bin=16384`
- `max_depth=4`
- `min_child_weight=10`
- `learning_rate=0.025`
- `reg_alpha=0.071`
- `reg_lambda=2`
- `subsample=1`
- `colsample_bytree=1`
- up to `20000` trees
- early stopping patience `500`
- seed `42`

This canonical notebook evaluates the selected configuration directly; it does not perform a parameter scan.

In [ ]:
MODEL_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    device=DEVICE,
    max_bin=MAX_BIN,

    max_depth=4,
    min_child_weight=10,
    learning_rate=0.025,

    n_estimators=ROUNDS,
    early_stopping_rounds=PATIENCE,

    reg_alpha=0.071,
    reg_lambda=2.0,

    subsample=1.0,
    colsample_bytree=1.0,

    random_state=SEED,
    n_jobs=THREADS,
    verbosity=0,
)

MODEL_PARAMS

## 8. Ten-fold OOF training

The outer validation structure is:

$$
\mathrm{StratifiedKFold}
(
10,
\mathrm{shuffle=True},
\mathrm{random\_state}=42
).
$$

One XGBoost model is trained per outer fold. Each fold gets its own frequency and Target-Encoding features.

The interaction groups are generated after outer-training constant-column removal, so their column indices always match the actual feature matrix passed to XGBoost.

In [ ]:
outer_cv = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

folds = list(
    outer_cv.split(
        train,
        y,
    )
)

fold_id = np.empty(
    len(train),
    dtype="int8",
)

for fold, (
    _,
    valid_idx,
) in enumerate(
    folds,
    start=1,
):
    fold_id[
        valid_idx
    ] = fold


oof_pred = np.full(
    len(train),
    np.nan,
    dtype="float64",
)

oof_rank = np.full(
    len(train),
    np.nan,
    dtype="float64",
)

test_pred = np.zeros(
    len(test),
    dtype="float64",
)

test_rank = np.zeros(
    len(test),
    dtype="float64",
)

fold_rows = []
importance_frames = []


def normalized_rank(values):
    values = np.asarray(
        values
    )

    return (
        rankdata(
            values,
            method="average",
        )
        - 0.5
    ) / len(values)


for fold, (
    train_idx,
    valid_idx,
) in enumerate(
    folds,
    start=1,
):
    (
        (
            X_train,
            X_valid,
            X_test,
        ),
        groups,
    ) = make_fold(
        train_idx,
        valid_idx,
    )

    params = MODEL_PARAMS.copy()
    params[
        "interaction_constraints"
    ] = json.dumps(
        groups
    )

    model = xgb.XGBClassifier(
        **params
    )

    model.fit(
        X_train,
        y[train_idx],
        eval_set=[
            (
                X_valid,
                y[valid_idx],
            )
        ],
        verbose=False,
    )

    valid_pred = (
        model.predict_proba(
            X_valid
        )[:, 1]
    )

    fold_test_pred = (
        model.predict_proba(
            X_test
        )[:, 1]
    )

    oof_pred[
        valid_idx
    ] = valid_pred

    oof_rank[
        valid_idx
    ] = normalized_rank(
        valid_pred
    )

    test_pred += (
        fold_test_pred
        / N_FOLDS
    )

    test_rank += (
        normalized_rank(
            fold_test_pred
        )
        / N_FOLDS
    )

    fold_auc = roc_auc_score(
        y[valid_idx],
        valid_pred,
    )

    best_iteration = int(
        model.best_iteration
        + 1
    )

    fold_rows.append({
        "fold": fold,
        "auc": fold_auc,
        "best_iteration": best_iteration,
        "n_features": X_train.shape[1],
        "n_interaction_groups": len(groups),
    })

    importance_frames.append(
        pd.DataFrame({
            "feature": X_train.columns,
            "gain": model.feature_importances_,
            "fold": fold,
        })
    )

    print(
        f"fold={fold}/{N_FOLDS} "
        f"AUC={fold_auc:.9f} "
        f"trees={best_iteration} "
        f"features={X_train.shape[1]} "
        f"groups={len(groups)}"
    )

## 9. Evaluate pooled OOF performance

The project's main comparison metric is pooled OOF AUC:

$$
\mathrm{AUC}_{\mathrm{OOF}}
=
\mathrm{AUC}
\left(
y,
\hat p_{\mathrm{OOF}}
\right).
$$

The fold-normalized rank vector is also retained because later ensemble research operated primarily in rank space.

In [ ]:
raw_oof_auc = roc_auc_score(
    y,
    oof_pred,
)

rank_oof_auc = roc_auc_score(
    y,
    oof_rank,
)

fold_metrics = pd.DataFrame(
    fold_rows
)

print(
    f"Raw OOF AUC:  {raw_oof_auc:.9f}"
)

print(
    f"Rank OOF AUC: {rank_oof_auc:.9f}"
)

print()
print(
    "Reproduction target:"
)
print(
    "  pooled raw OOF ≈ 0.946215002"
)

## 10. Feature importance

XGBoost gain-style feature importances are averaged across the ten outer-fold models.

This is descriptive only; the notebook does not select features from validation importance.

In [ ]:
importance_all = pd.concat(
    importance_frames,
    ignore_index=True,
)

importance_summary = (
    importance_all
    .groupby(
        "feature",
        as_index=False,
    )["gain"]
    .mean()
    .sort_values(
        "gain",
        ascending=False,
    )
)

importance_summary.head(30)

## 11. Save reproducible outputs

The output schema is intentionally compatible with the later ensemble notebooks.

In [ ]:
oof_output = pd.DataFrame({
    ID_COL: train[ID_COL],
    "y_true": y,
    "fold": fold_id,
    "prediction": oof_pred,
    "prediction_rank": oof_rank,
})

test_output = pd.DataFrame({
    ID_COL: test[ID_COL],
    "prediction": test_pred,
    "prediction_rank": test_rank,
})

submission = sample_submission.copy()
submission[TARGET] = test_pred

oof_output.to_csv(
    OUTPUT_DIR
    / "xgb_additive_oof_predictions.csv",
    index=False,
)

test_output.to_csv(
    OUTPUT_DIR
    / "xgb_additive_test_predictions.csv",
    index=False,
)

fold_metrics.to_csv(
    OUTPUT_DIR
    / "fold_metrics.csv",
    index=False,
)

importance_summary.to_csv(
    OUTPUT_DIR
    / "feature_importance.csv",
    index=False,
)

submission.to_csv(
    OUTPUT_DIR
    / "submission_additive_xgb.csv",
    index=False,
)

print(
    "Saved outputs to:",
    OUTPUT_DIR,
)

## 12. Interpretation

This model is important because the improvement did not come from simply increasing model capacity.

The experiment combined:

- high histogram resolution (`max_bin=16384`);
- strong income representation;
- fold-local frequency features;
- cross-fitted supervised encodings;
- shallow trees;
- strong field-level interaction constraints.

The resulting model achieved OOF AUC around **0.946215**, slightly below the strongest V2 blend as a standalone predictor, but its prediction ordering was complementary enough to improve the ensemble materially.

A later weight scan found a broad plateau around 40–55% additive-XGB weight. The project selected 40%, producing the four-model internal champion:

$$
0.40\,R_{\mathrm{XGB}}
+
0.24\,R_{\mathrm{LGB\ exact}}
+
0.24\,R_{\mathrm{CatBoost}}
+
0.12\,R_{\mathrm{LGB\ coarse}},
$$

with OOF AUC approximately **0.946335057**.

That ensemble result is analyzed in the final research notebook rather than here.

## Result

- Raw OOF AUC: **0.946215002**
- Rank-normalized OOF AUC: **0.946214822**

This component is used in the final ensemble analysis in
`06_final_research.ipynb`.